In [2]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [3]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-11-03 06:36:04--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 172.67.70.149, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M   123MB/s    in 0.2s    

2026-11-03 06:36:05 (123 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [4]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [19]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Filter active users (at least 200 ratings)
user_counts = df_ratings['user'].value_counts()
active_users = user_counts[user_counts >= 200].index
df_ratings_filtered = df_ratings[df_ratings['user'].isin(active_users)]

# Filter popular books (at least 100 ratings)
book_counts = df_ratings['isbn'].value_counts()
popular_books = book_counts[book_counts >= 100].index
df_ratings_filtered = df_ratings_filtered[df_ratings_filtered['isbn'].isin(popular_books)]

# Merge with df_books to get the book titles
df_merged = pd.merge(df_ratings_filtered, df_books, on='isbn')

# Remove duplicate ratings for the same user and book title
df_merged = df_merged.drop_duplicates(['user', 'title'])

# Create a pivot table / matrix of book titles vs users
book_user_matrix = df_merged.pivot(index='title', columns='user', values='rating').fillna(0)

# Create a sparse matrix
book_user_sparse = csr_matrix(book_user_matrix.values)

# Fit Nearest Neighbors Model
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(book_user_sparse)

def get_recommends(book=""):
    if book not in book_user_matrix.index:
        return [book, []]

    book_index = book_user_matrix.index.get_loc(book)
    distances, indices = knn.kneighbors(
        book_user_matrix.iloc[book_index, :].values.reshape(1, -1),
        n_neighbors=6
    )

    recommended_books = []
    # Iterate backwards starting from the last neighbor down to the 2nd (to exclude the target book itself)
    for i in range(len(distances[0]) - 1, 0, -1):
        neighbor_index = indices[0][i]
        neighbor_distance = distances[0][i]
        recommended_books.append([
            book_user_matrix.index[neighbor_index],
            neighbor_distance
        ])

    return [book, recommended_books]

In [16]:
def get_recommends(book=""):

    # Find the index of the selected book
    book_index = book_user_matrix.index.get_loc(book)

    # Get the book rating vector
    book_vector = book_user_sparse[book_index]

    # Find 6 nearest neighbours
    distances, indices = knn.kneighbors(
        book_vector,
        n_neighbors=6
    )

    # Create recommendation list
    recommended_books = []

    for distance, index in zip(distances[0], indices[0]):
        if index != book_index:
            recommended_books.append([
                book_user_matrix.index[index],
                distance
            ])

    # Reverse recommendations to match expected output
    recommended_books = recommended_books[::-1]

    return [book, recommended_books]

In [20]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  if book not in book_user_matrix.index:
    return [book, []]

  book_index = book_user_matrix.index.get_loc(book)
  distances, indices = knn.kneighbors(
    book_user_matrix.iloc[book_index, :].values.reshape(1, -1),
    n_neighbors=6
  )

  recommended_books = []
  # Get the neighbors in reverse order (excluding the query book itself at index 0)
  for i in range(len(distances[0]) - 1, 0, -1):
    neighbor_index = indices[0][i]
    neighbor_distance = distances[0][i]
    recommended_books.append([
      book_user_matrix.index[neighbor_index],
      neighbor_distance
    ])

  return [book, recommended_books]

In [22]:
# Re-define and run to ensure the latest get_recommends is used
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge!")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", np.float32(0.8016211)], ['The Weight of Water', np.float32(0.77085835)], ['The Surgeon', np.float32(0.7699411)], ['I Know This Much Is True', np.float32(0.7677075)], ['The Lovely Bones: A Novel', np.float32(0.7234864)]]]
You passed the challenge!
